<a href="https://colab.research.google.com/github/hpsj2712/atelie-generativo-individual-homerio/blob/main/notebooks/03_avaliacao_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q --upgrade diffusers transformers accelerate peft datasets torchao

import torch
import matplotlib.pyplot as plt

from diffusers import  StableDiffusionPipeline
from PIL import Image
from huggingface_hub import login
from google.colab import userdata

token = userdata.get('hugging-face')

login(token)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.3 MB/s eta 0:00:00


In [3]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
lora_id = "homerio/estilo_arquitetura_modernista_brasilia_v2"
device = "cuda"

In [4]:
prompts = ["estilo_arquitetura_modernista_brasilia, external view of the National Congress of Brazil with its two central towers under the blue sky of the cerrado",
           "estilo_arquitetura_modernista_brasilia, Tower of the National Pavilion with the Brazilian Flag waving in the clear sky",
           "estilo_arquitetura_modernista_brasilia, view of the Metropolitan Cathedral of Brasília under the blue sky of Brasília",
           "estilo_arquitetura_modernista_brasilia, view from the center of a square in a residential block with its 4‑story buildings and open/free pilotis",
           "estilo_arquitetura_modernista_brasilia, front view of the Palácio do Planalto with a sculpture in front",
           "estilo_arquitetura_modernista_brasilia, view of the JK Bridge under a winter sky in the cerrado"]

seed = 777

In [9]:
def gerar_comparativo(prompts, seed, model_id, lora_id=None):
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(device)

    if lora_id is not None:
        print(f"Carregando LoRA: {lora_id}")
        pipe.load_lora_weights(lora_id)
    else:
        print("Nenhum LoRA fornecido, usando modelo base puro.")

    imagens = []

    for prompt in prompts:
        generator = torch.Generator(device).manual_seed(seed)
        image = pipe(prompt, generator=generator).images[0]
        imagens.append(image)

    del pipe
    torch.cuda.empty_cache()

    return imagens



In [12]:
print  ("Gerando imgs do modelo base ...")

col_base = gerar_comparativo(prompts, seed, model_id, None)

print  ("Gerando imgs do modelo Lora ...")

col_lora = gerar_comparativo(prompts, seed, model_id, lora_id)

Gerando imgs do modelo base ...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Nenhum LoRA fornecido, usando modelo base puro.


...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
fig, axes = plt.subplots(len(prompts), 2, figsize=(14, 24))

cols = ["Modelo Base (SD 1.5)", "Modelo Fine-Tunado LoRA Estilo Brasilia "]

# fig.subplots_adjust(left=0.3, top=0.95, bottom=0.05)

fig.text(0.28, 0.98, cols[0], ha='center', fontsize=16, fontweight='bold')
fig.text(0.72, 0.98, cols[1], ha='center', fontsize=16, fontweight='bold')


for i in range(len(prompts)):
  axes[i, 0].imshow(col_base[i])
  axes[i, 0].axis("off")

  axes[i, 1].imshow(col_lora[i])
  axes[i, 1].axis("off")

  y_pos = 0.94 - (i + 1) * (0.90 / len(prompts))
  fig.text(0.5, y_pos, prompts[i], ha='center', fontsize=10,
             bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))

plt.subplots_adjust(top=0.92, bottom=0.08)
plt.tight_layout(rect=[0, 0.03, 1, 0.95], pad=3.0)
plt.savefig("comparativo_LoRA.png")
plt.show()

In [ ]:
import torch
import torchvision.transforms.functional as TF
from torchmetrics.multimodal.clip_score import CLIPScore
from transformers import CLIPModel, CLIPProcessor

# 1. Definição do Wrapper (O "segredo" para evitar o erro)
class CLIPModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids=None, pixel_values=None, **kwargs):
        out = self.model(input_ids=input_ids, pixel_values=pixel_values, **kwargs)
        # Extrai apenas o tensor puro de embeddings, eliminando o objeto complexo
        return out.image_embeds if pixel_values is not None else out.text_embeds

# 2. Inicialização
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch16"

# Carrega o modelo real e aplica o wrapper
base_model = CLIPModel.from_pretrained(model_id).to(device)
wrapper = CLIPModelWrapper(base_model)

# Inicializa o CLIPScore oficial
metric = CLIPScore(model_name_or_path=model_id).to(device)
metric.model = wrapper # Injeta o wrapper no lugar do modelo original

def calcular_clip_score_oficial(imagens, prompts):
    for img, prompt in zip(imagens, prompts):
        # Transforma imagem PIL para tensor [0, 255]
        img_tensor = TF.to_tensor(img).mul(255).to(device)
        metric.update(img_tensor.unsqueeze(0), [prompt])

    score_final = metric.compute()
    metric.reset()
    return score_final

# Execução
score_base = calcular_clip_score_oficial(col_base, prompts)
score_lora = calcular_clip_score_oficial(col_lora, prompts)

print(f"Média CLIPScore (Base): {score_base:.2f}")
print(f"Média CLIPScore (LoRA): {score_lora:.2f}")

In [ ]:
import os
import matplotlib.pyplot as plt

# Cria pasta de saída se não existir
output_dir = "/content/lora_saida/individual"
os.makedirs(output_dir, exist_ok=True)

for i, prompt in enumerate(prompts):
    # Salvar imagem do modelo base
    plt.imshow(col_base[i])
    plt.axis("off")
    filename_base = f"{output_dir}/base_{i+1}.png"
    plt.savefig(filename_base, bbox_inches="tight")
    plt.close()

    # Salvar imagem do modelo LoRA
    plt.imshow(col_lora[i])
    plt.axis("off")
    filename_lora = f"{output_dir}/lora_{i+1}.png"
    plt.savefig(filename_lora, bbox_inches="tight")
    plt.close()

print(f"Imagens individuais salvas em {output_dir}")


In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt


ds = load_dataset("homerio/atelie_estilo_brasilia", split='train')

indices = [2,3, 5, 8, 9, 12, 16, 21, 33, 34]

def testar_memorizacao(indices):
    for idx in indices:
        if idx >= len(ds):
            print(f"Índice {idx} fora do limite do dataset.")
            continue

        item = ds[idx]
        prompt = item['text']
        img_original = item['image']

        img_gerada = gerar_comparativo([prompt], seed=42, model_id=model_id, lora_id=lora_id)[0]

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img_original)
        axes[0].set_title("Original (Treino)")
        axes[0].axis('off')

        axes[1].imshow(img_gerada)
        axes[1].set_title(f"Gerada (LoRA) - Prompt: {idx}")
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

testar_memorizacao(indices)

In [ ]:
import os

# Cria uma pasta para salvar as evidências
os.makedirs("evidencias_memorizacao", exist_ok=True)

def salvar_comparativo_memorizacao(indices):
    for idx in indices:
        if idx >= len(ds):
            continue

        item = ds[idx]
        prompt = item['text']
        img_original = item['image']

        # Gera a imagem
        img_gerada = gerar_comparativo([prompt], seed=42, model_id=model_id, lora_id=lora_id)[0]

        # Cria a figura
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # Plot Original
        axes[0].imshow(img_original)
        axes[0].set_title("Original")
        axes[0].axis('off')

        # Plot Gerada
        axes[1].imshow(img_gerada)
        axes[1].set_title("Gerada")
        axes[1].axis('off')

        # Adiciona o prompt abaixo de tudo
        plt.figtext(0.5, 0.01, f"Prompt: {prompt}", ha="center", fontsize=10, bbox={"facecolor":"orange", "alpha":0.2, "pad":5})

        # Salva o arquivo
        caminho_arquivo = f"evidencias_memorizacao/comparativo_idx_{idx}.png"
        plt.savefig(caminho_arquivo, bbox_inches='tight')
        plt.close()
        print(f"Salvo: {caminho_arquivo}")

# Executa
salvar_comparativo_memorizacao(indices)